# Exp2-v2 — BoW + 전처리 + 핸드크래프트 + W&B Sweep

- 목적: Exp2(BoW)에서 Exp3 개선 아이디어(전처리, handcraft feature)를 이식해 성능 향상 확인
- 제약: MLP 구조 유지

In [1]:
!pip install datasets wandb scikit-learn -q

In [2]:
import re, copy, numpy as np
import torch, torch.nn as nn, torch.optim as optim, torch.backends.cudnn as cudnn
from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score
from scipy.sparse import hstack, csr_matrix
import wandb

SEED=42
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
cudnn.benchmark=False; cudnn.deterministic=True
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:{device}')

Device:cuda


In [3]:
data=load_dataset('Sp1786/multiclass-sentiment-analysis-dataset')
def remove_empty(row):
    return all(row[f] not in [None,''] for f in ['id','text','label','sentiment'])
train_data=data['train'].filter(remove_empty)
dev_data=data['validation'].filter(remove_empty)
test_data=data['test'].filter(remove_empty)
output_size=len(set(train_data['label']))
train_labels=train_data['label']
test_labels_list=test_data['label']
print(f"Train:{len(train_data)} | Dev:{len(dev_data)} | Test:{len(test_data)} | Classes:{output_size}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train_df.csv: 0.00B [00:00, ?B/s]

val_df.csv: 0.00B [00:00, ?B/s]

test_df.csv: 0.00B [00:00, ?B/s]

...

Generating test split:   0%|          | 0/5206 [00:00<?, ? examples/s]

Filter:   0%|          | 0/31232 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5205 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5206 [00:00<?, ? examples/s]

Train:31232 | Dev:5205 | Test:5205 | Classes:3


In [4]:
def preprocess_text(text):
    text = str(text).lower()
    text = text.replace('`', "'")
    text = text.replace('****', ' bad ').replace('***', ' bad ')
    text = re.sub(r"won't", "will not", text)
    text = re.sub(r"can't", "cannot", text)
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"'re", " are", text)
    text = re.sub(r"'ve", " have", text)
    text = re.sub(r"'ll", " will", text)
    text = re.sub(r"'d", " would", text)
    text = re.sub(r"'m", " am", text)
    for pat, rep in [
        (r'\\bidk\\b','i do not know'), (r'\\bur\\b','your'),
        (r'\\bnaw\\b','no'), (r'\\bgonna\\b','going to'),
        (r'\\bwanna\\b','want to'), (r'\\blol\\b','laughing'),
        (r'\\bomg\\b','oh my god'), (r'\\bwtf\\b','what the'),
    ]:
        text = re.sub(pat, rep, text)
    return text

def extract_handcraft(texts):
    feats=[]
    for text in texts:
        t=str(text)
        tl=t.lower()
        words=t.split()
        feats.append([
            min(t.count('!'),5),
            min(t.count('?'),5),
            sum(1 for w in words if w.isupper() and len(w)>1),
            min(len(words),50),
            int('http' in tl),
            int(any(e in tl for e in [':)',':(',':d',':/','haha','hehe','lmao']))
        ])
    return np.array(feats, dtype=np.float32)

vectorizer=CountVectorizer(max_features=30000, preprocessor=preprocess_text, min_df=2)
vectorizer.fit(train_data['text'])

def build_features(split):
    bow=vectorizer.transform(split['text'])
    hc=csr_matrix(extract_handcraft(split['text']))
    x=hstack([bow,hc])
    return torch.FloatTensor(x.toarray()).to(device)

train_t=build_features(train_data)
dev_t=build_features(dev_data)
test_t=build_features(test_data)
dev_labels_t=torch.tensor(dev_data['label'],dtype=torch.long).to(device)
input_size=train_t.shape[1]
print(f'Input size: {input_size}')

Input size: 11778


In [5]:
class MLP(nn.Module):
    def __init__(self,i,h,o,d=0.0):
        super().__init__()
        self.fc1=nn.Linear(i,h)
        self.fc2=nn.Linear(h,h//2)
        self.fc3=nn.Linear(h//2,o)
        self.activation=nn.GELU()
        self.output_act=nn.Softmax(dim=1)
        self.dropout=nn.Dropout(p=d)
    def forward(self,x):
        x=self.dropout(self.activation(self.fc1(x)))
        x=self.dropout(self.activation(self.fc2(x)))
        return self.output_act(self.fc3(x))

In [6]:
sweep_config={
    'method':'bayes',
    'metric':{'name':'best_dev_accuracy','goal':'maximize'},
    'parameters':{
        'hidden_size':{'values':[256,512,1000]},
        'learning_rate':{'distribution':'log_uniform_values','min':1e-5,'max':1e-3},
        'dropout':{'values':[0.1,0.2,0.3,0.4]},
        'weight_decay':{'values':[0,1e-5,1e-4]},
        'batch_size':{'values':[128,256]},
        'num_epochs':{'values':[30,50]},
    }
}
sweep_id=wandb.sweep(sweep_config, project='nlp-hw1')
print(f'Sweep ID: {sweep_id}')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Create sweep with ID: t0mtsry4
Sweep URL: https://wandb.ai/imeanseo_/nlp-hw1/sweeps/t0mtsry4
Sweep ID: t0mtsry4


In [7]:
def train_sweep():
    run=wandb.init()
    cfg=run.config
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model=MLP(input_size,cfg.hidden_size,output_size,cfg.dropout).to(device)
    opt=optim.Adam(model.parameters(),lr=cfg.learning_rate,weight_decay=cfg.weight_decay)
    lfn=nn.CrossEntropyLoss()
    best_dev,best_state=0,None

    for epoch in range(cfg.num_epochs):
        model.train()
        total_loss=0
        for i in range(0,len(train_t),cfg.batch_size):
            bd=train_t[i:i+cfg.batch_size]
            bl=torch.tensor(train_labels[i:i+cfg.batch_size],device=device)
            loss=lfn(model(bd),bl)
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss+=loss.item()

        model.eval()
        with torch.no_grad():
            da=(torch.argmax(model(dev_t),dim=1)==dev_labels_t).float().mean().item()
        if da>best_dev:
            best_dev,best_state=da,copy.deepcopy(model.state_dict())

        wandb.log({
            'epoch':epoch+1,
            'train_loss':total_loss/max(1,len(train_t)//cfg.batch_size),
            'dev_accuracy':da,
            'best_dev_accuracy':best_dev,
        })

    model.load_state_dict(best_state)
    with torch.no_grad():
        test_acc=accuracy_score(test_labels_list,torch.argmax(model(test_t),dim=1).cpu().tolist())
    wandb.log({'test_accuracy':test_acc})
    print(f"[Exp2-v2] Dev:{best_dev:.4f}|Test:{test_acc*100:.2f}%")
    wandb.finish()

wandb.agent(sweep_id, train_sweep, count=12)

wandb: Agent Starting Run: qm01y4rv with config:
wandb: 	batch_size: 256
wandb: 	dropout: 0.3
wandb: 	hidden_size: 256
wandb: 	learning_rate: 4.4344291997607416e-05
wandb: 	num_epochs: 30
wandb: 	weight_decay: 1e-05
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: aileen02-ko (imeanseo_) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[Exp2-v2] Dev:0.6939|Test:68.72%


best_dev_accuracy,▁▂▄▅▆▆▇▇▇▇▇▇▇█████████████████
dev_accuracy,▁▂▄▅▆▆▇▇▇▇▇▇▇█████████████████
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
test_accuracy,▁
train_loss,███▇▆▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
best_dev_accuracy,0.69395
dev_accuracy,0.68934
epoch,30
test_accuracy,0.68722
train_loss,0.77679


wandb: Agent Starting Run: 9ikm16av with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0.1
wandb: 	hidden_size: 1000
wandb: 	learning_rate: 3.6264130908612174e-05
wandb: 	num_epochs: 50
wandb: 	weight_decay: 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


[Exp2-v2] Dev:0.6930|Test:68.59%


...

[Exp2-v2] Dev:0.6964|Test:68.70%


best_dev_accuracy,▁▂▄▄▅▆▆▇▇▇▇▇▇▇██████████████████████████
dev_accuracy,▁▄▄▅▆▆▇▇▇▇▇▇▇███████████████████████████
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
test_accuracy,▁
train_loss,████▇▆▅▅▅▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
best_dev_accuracy,0.69645
dev_accuracy,0.6928
epoch,50
test_accuracy,0.68703
train_loss,0.73284


wandb: Agent Starting Run: k518crtb with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0.1
wandb: 	hidden_size: 1000
wandb: 	learning_rate: 0.00017791830878562768
wandb: 	num_epochs: 30
wandb: 	weight_decay: 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


[Exp2-v2] Dev:0.6807|Test:67.92%


best_dev_accuracy,▁▄▄███████████████████████████
dev_accuracy,▂▅▁██▆▅▅▆▆▆▆▆▆▆▅▅▆▄▄▃▄▄▅▅▅▅▄▆▆
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
test_accuracy,▁
train_loss,█▅▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_dev_accuracy,0.68069
dev_accuracy,0.66955
epoch,30
test_accuracy,0.67915
train_loss,0.67981


In [8]:
USERNAME='imeanseo_'
api=wandb.Api()
sw=api.sweep(f'{USERNAME}/nlp-hw1/{sweep_id}')
best=sw.best_run()
print('\n'+'='*60)
print('🏆 Best Run Config:')
for k,v in dict(best.config).items():
    print(f'  {k:<20}: {v}')
print(f"\nBest Dev:{best.summary['best_dev_accuracy']:.4f}")
print(f"Test:{best.summary['test_accuracy']*100:.2f}%")
print('='*60)

wandb: Sorting runs by -summary_metrics.best_dev_accuracy



🏆 Best Run Config:
  dropout             : 0.3
  batch_size          : 256
  num_epochs          : 50
  hidden_size         : 1000
  weight_decay        : 0
  learning_rate       : 1.600488608953689e-05

Best Dev:0.6964
Test:68.70%


In [10]:
import random # Added to fix NameError

# ⚠️ 위 출력 값으로 수정
BEST_H, BEST_LR, BEST_D, BEST_WD, BEST_EP, BEST_BS = 1000, 1.600488608953689e-05, 0.3, 0, 50, 256

# Seed 완벽 재고정
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

final=MLP(input_size,BEST_H,output_size,BEST_D).to(device)
opt=optim.Adam(final.parameters(),lr=BEST_LR,weight_decay=BEST_WD)
lfn=nn.CrossEntropyLoss()
best_dev,best_state=0,None
for epoch in range(BEST_EP):
    final.train()
    for i in range(0,len(train_t),BEST_BS):
        bd=train_t[i:i+BEST_BS]
        bl=torch.tensor(train_labels[i:i+BEST_BS],device=device)
        loss=lfn(final(bd),bl)
        opt.zero_grad(); loss.backward(); opt.step()
    final.eval()
    with torch.no_grad():
        da=(torch.argmax(final(dev_t),dim=1)==dev_labels_t).float().mean().item()
    if da>best_dev:
        best_dev,best_state=da,copy.deepcopy(final.state_dict())
    print(f'Epoch {epoch+1}/{BEST_EP}|Dev:{da:.4f}')
final.load_state_dict(best_state)
torch.save(best_state,'best_model_exp2_v2.pt')
with torch.no_grad():
    test_acc=accuracy_score(test_labels_list,torch.argmax(final(test_t),dim=1).cpu().tolist())
print(f'\n✅ 저장:best_model_exp2_v2.pt|Dev:{best_dev:.4f}|Test:{test_acc*100:.2f}%')

Epoch 1/50|Dev:0.3437
Epoch 2/50|Dev:0.4106
Epoch 3/50|Dev:0.4745
Epoch 4/50|Dev:0.4999
Epoch 5/50|Dev:0.5524
Epoch 6/50|Dev:0.5800
Epoch 7/50|Dev:0.5950
Epoch 8/50|Dev:0.6158
Epoch 9/50|Dev:0.6307
Epoch 10/50|Dev:0.6398
Epoch 11/50|Dev:0.6486
Epoch 12/50|Dev:0.6555
Epoch 13/50|Dev:0.6607
Epoch 14/50|Dev:0.6642
Epoch 15/50|Dev:0.6671
Epoch 16/50|Dev:0.6703
Epoch 17/50|Dev:0.6747
Epoch 18/50|Dev:0.6755
Epoch 19/50|Dev:0.6770
Epoch 20/50|Dev:0.6770
Epoch 21/50|Dev:0.6820
Epoch 22/50|Dev:0.6840
Epoch 23/50|Dev:0.6841
Epoch 24/50|Dev:0.6813
Epoch 25/50|Dev:0.6824
Epoch 26/50|Dev:0.6820
Epoch 27/50|Dev:0.6853
Epoch 28/50|Dev:0.6872
Epoch 29/50|Dev:0.6890
Epoch 30/50|Dev:0.6880
Epoch 31/50|Dev:0.6880
Epoch 32/50|Dev:0.6884
Epoch 33/50|Dev:0.6888
Epoch 34/50|Dev:0.6913
Epoch 35/50|Dev:0.6897
Epoch 36/50|Dev:0.6922
Epoch 37/50|Dev:0.6924
Epoch 38/50|Dev:0.6922
Epoch 39/50|Dev:0.6930
Epoch 40/50|Dev:0.6936
Epoch 41/50|Dev:0.6938
Epoch 42/50|Dev:0.6932
Epoch 43/50|Dev:0.6949
Epoch 44/50|Dev:0.69